In [906]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.ess import compute_ess

In [907]:
sns.set_style("darkgrid")

In [908]:
# the run outputs live in the tests directory at the repository root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tests" / "hcv_emat.xml").exists())
TESTS_PATH = ROOT / "tests"

In [909]:
method_names = ["emat", "mix_3", "mix_6", "mix_9", "beauti"]

# the wall-clock seconds per million MCMC samples of each run, which convert samples into running time
seconds_per_million_samples = {
    "hcv": {"emat": 12.0, "mix": 14.0, "beauti": 23.0, "mix_3": 18.0, "mix_6": 16.0, "mix_9": 14.0,},
    "zika": {"emat": 27.0, "mix": 73.0, "beauti": 394.0},
    "sars": {"emat": 115.0, "beauti": 6200.0},
}

In [910]:
experiment = "hcv"

In [911]:
# likelihood columns are left out, as the EMAT regimes log the augmented likelihood
columns_per_experiment = {
    "hcv": ["posterior", "Tree.height", "Tree.treeLength", "rateAG", "freqParameter.1", "bPopSizes.1", "bGroupSizes.2", "BayesianSkyline"],
    "zika": ["posterior", "Tree.height", "Tree.treeLength", "clockRate", "kappa", "freqParameter.1", "CoalescentExponential", "ePopSize", "growthRate"],
    "sars": ["posterior", "Tree.height", "Tree.treeLength", "clockRate", "kappa", "freqParameter.1", "CoalescentExponential", "ePopSize", "growthRate"],
}

In [912]:
def get_ess_df(log_df: pd.DataFrame, column: str, method: str, num_points: int = 250):
    ess_dict = {
        "sample": [],
        "seconds": [],
        "ess": [],
        "ess_per_second": [],
        "column": [],
        "method": [],
    }

    burnin = len(log_df) // 5
    samples = log_df["Sample"].values[burnin:]
    col_values = log_df[column].values[burnin:]

    step = max(1, len(col_values) // num_points)
    seconds_per_sample = seconds_per_million_samples[experiment][method] / 1e6

    for i in range(step, len(col_values) + 1, step):
        # estimate ESS for the first i logged values after burn-in
        try:
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ess_val = compute_ess(col_values[:i])

            ess_dict["sample"].append(samples[i - 1])
            seconds = samples[i - 1] * seconds_per_sample
            ess_dict["seconds"].append(seconds)
            ess_dict["ess"].append(ess_val)
            ess_dict["ess_per_second"].append(ess_val / seconds)
            ess_dict["column"].append(column)
            ess_dict["method"].append(method)
        except Exception:
            pass

    return pd.DataFrame(ess_dict)

In [ ]:
ess_dfs = []

for method in method_names:
    # the log files are named <experiment>_<method>.log or <experiment>_<method>-<seed>.log
    log_paths = sorted(TESTS_PATH.glob(f"{experiment}_{method}.log")) + sorted(TESTS_PATH.glob(f"{experiment}_{method}-*.log"))
    if not log_paths:
        print(f"no log found for {experiment}_{method}")

    log_dfs = [pd.read_csv(path, comment="#", delimiter="\t") for path in log_paths]

    for column in columns_per_experiment[experiment]:
        for log_df in log_dfs:
            if column in log_df.columns:
                ess_dfs.append(get_ess_df(log_df, column, method))
            else:
                print(f"{experiment}_{method}: column {column} not logged")

ess_df = pd.concat(ess_dfs)

columns = columns_per_experiment[experiment]
num_cols = 3
num_rows = (len(columns) + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(3 * num_cols, 2 * num_rows), squeeze=False)

for ax, column in zip(axes.flat, columns):
    # drop duplicate index labels that may cause seaborn to fail
    data = ess_df[ess_df.column == column].reset_index(drop=True)
    sns.lineplot(data, x="seconds", y="ess", hue="method", hue_order=method_names, ax=ax, legend=ax is axes.flat[0])
    ax.set_title(column)
    ax.set_xlabel("running time (s)")
    ax.set_ylabel("ESS")

for ax in axes.flat[len(columns):]:
    ax.set_visible(False)

fig.suptitle(experiment)
fig.tight_layout()
plt.show()